# Model Training Notebook

This notebook handles two scenarios:

1. **Training a new model** (default): Creates a new time series forecasting model from scratch
2. **Using an existing deployment**: Extracts metadata from an existing DataRobot deployment

To use an existing deployment, set the `FORECAST_DEPLOYMENT_ID` environment variable in your `.env` file:
```
FORECAST_DEPLOYMENT_ID=your-deployment-id-here
```

When using an existing deployment, the notebook will:
- Skip data ingestion and model training
- Extract model metadata from the existing deployment
- Generate the same app configuration files needed for the frontend

**Note**: When using an existing deployment, you may need to adjust the `feature_settings_config` in this notebook to match your model's known-in-advance features.

In [ ]:
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

import datarobot as dr
from dotenv import load_dotenv

# The notebook should be executed from the project root directory
if "_correct_path" not in locals():
    os.chdir("..")
    sys.path.append(".")
    print(f"changed dir to {Path('.').resolve()})")
    _correct_path = True
load_dotenv()
client = dr.Client()

In [ ]:
from datarobotx.idp.use_cases import get_or_create_use_case

from infra.settings_main import use_case_args

if "DATAROBOT_DEFAULT_USE_CASE" in os.environ:
    use_case_id = os.environ["DATAROBOT_DEFAULT_USE_CASE"]
else:
    use_case_id = get_or_create_use_case(
        endpoint=client.endpoint,
        token=client.token,
        name=use_case_args.resource_name,
        description=use_case_args.description,
    )

In [ ]:
# Check if we should use an existing deployment instead of training a new model
FORECAST_DEPLOYMENT_ID = os.environ.get("FORECAST_DEPLOYMENT_ID")

if FORECAST_DEPLOYMENT_ID:
    print(f"Using existing deployment: {FORECAST_DEPLOYMENT_ID}")
    print("Skipping model training and using existing deployment metadata...")

    deployment = dr.Deployment.get(FORECAST_DEPLOYMENT_ID)
    model = deployment.model
    project_id = model["project_id"]
    model_id = model["id"]

    # model_package contains both the version ID and the parent registered model ID directly
    registered_model_version_id = deployment.model_package["id"]
    registered_model_id = deployment.model_package.get("registered_model_id")
    registered_model_name = deployment.model_package.get(
        "name", f"Model Package {registered_model_version_id}"
    )

    print(f"Found model: {model_id} in project: {project_id}")
    print(f"Registered model ID:      {registered_model_id}")
    print(f"Registered model version: {registered_model_version_id}")
    print(f"Registered model name:    {registered_model_name}")

    SKIP_TRAINING = True
else:
    print("No existing deployment specified. Will train a new model...")
    SKIP_TRAINING = False

# Data Ingest and Preparation
(Only executed when training a new model)

In [ ]:
TRAINING_DATASET_ID = os.environ.get("TRAINING_DATASET_ID")

if not SKIP_TRAINING:
    if TRAINING_DATASET_ID:
        print(f"Using existing training dataset: {TRAINING_DATASET_ID}")
        training_dataset_id = TRAINING_DATASET_ID
    else:
        import pandas as pd

        from infra.settings_datasets import training_dataset

        # Replace as needed with your own data ingest and/or preparation logic
        df = pd.read_csv(training_dataset.file_path)
else:
    print("Skipping data ingest - using existing deployment")

In [ ]:
if not SKIP_TRAINING and not TRAINING_DATASET_ID:
    from datarobotx.idp.datasets import get_or_create_dataset_from_df

    print("Uploading training data to AI Catalog...")
    training_dataset_id = get_or_create_dataset_from_df(
        endpoint=client.endpoint,
        token=client.token,
        data_frame=df,
        name=training_dataset.resource_name,
        use_cases=use_case_id,
    )
elif SKIP_TRAINING:
    print("Skipping dataset upload - using existing deployment")

# Model Training
(Only executed when training a new model)

In [ ]:
if not SKIP_TRAINING:
    from datarobot_pulumi_utils.schema.training import (
        AdvancedOptionsArgs,
        AnalyzeAndModelArgs,
        AutopilotRunArgs,
        CalendarArgs,
        DatetimePartitioningArgs,
    )

    from forecastic.schema import FeatureSettingConfig
    from infra.settings_main import project_name

    calendar_args = CalendarArgs(
        country_code="US",
        name=f"Calendar [{project_name}]",
        start_date="2012-01-01",
        end_date="2022-01-01",
    )
    autopilotrun_args = AutopilotRunArgs(
        name=f"Forecast Assistant Project [{project_name}]",
        advanced_options_config=AdvancedOptionsArgs(seed=42),
        analyze_and_model_config=AnalyzeAndModelArgs(
            metric="RMSE",
            mode=dr.enums.AUTOPILOT_MODE.QUICK,
            target="Sales",
            worker_count=-1,
            max_wait=1200,
        ),
        datetime_partitioning_config=DatetimePartitioningArgs(
            datetime_partition_column="Date",
            multiseries_id_columns=["Store"],
            use_time_series=True,
            feature_derivation_window_start=-35,
            feature_derivation_window_end=0,
            forecast_window_start=1,
            forecast_window_end=30,
        ),
        feature_settings_config=[
            FeatureSettingConfig(feature_name="Store_Size", known_in_advance=True),
            FeatureSettingConfig(feature_name="Marketing", known_in_advance=True),
            FeatureSettingConfig(feature_name="TouristEvent", known_in_advance=True),
        ],
    )

    registered_model_name = f"Forecastic Registered Model [{project_name}]"
else:
    print("Skipping training configuration - using existing deployment")

In [ ]:
from datarobotx.idp.registered_model_versions import (
    get_or_create_registered_leaderboard_model_version,
)

if not SKIP_TRAINING:
    from datarobotx.idp.autopilot import get_or_create_autopilot_run
    from datarobotx.idp.calendars import (
        get_or_create_calendar_dataset_from_country_code,
    )

    calendar_id = get_or_create_calendar_dataset_from_country_code(
        endpoint=client.endpoint, token=client.token, **calendar_args.model_dump()
    )

    print("Running Autopilot...")
    project_id = get_or_create_autopilot_run(
        endpoint=client.endpoint,
        token=client.token,
        calendar_id=calendar_id,
        dataset_id=training_dataset_id,
        use_case=use_case_id,
        **autopilotrun_args.model_dump(),
    )

    model_id = dr.ModelRecommendation.get(project_id).model_id
else:
    print("Skipping training - using existing deployment model")

# Always register a fresh model package with compute_all_ts_intervals=True. A deployment
# can only enable prediction interval settings if the model package it was created from
# was registered with that flag - reusing an existing deployment's own model_package (as
# done above when SKIP_TRAINING) doesn't work, because that package was never registered
# that way, even though the underlying model already has intervals calculated.
print("Registering model version (with prediction intervals)...")
registered_model_version_id = get_or_create_registered_leaderboard_model_version(
    endpoint=client.endpoint,
    token=client.token,
    model_id=model_id,
    registered_model_name=registered_model_name,
    compute_all_ts_intervals=True,
)
# get_or_create_registered_leaderboard_model_version only returns the version ID; look up
# its parent registered model directly (a plain model package read, same permission level
# as deployment.model_package above) rather than assuming it matches whatever
# registered_model_id was extracted from an existing deployment, since a new registered
# model may have just been created for it.
registered_model_id = client.get(
    f"modelPackages/{registered_model_version_id}/"
).json()["registeredModelId"]

# Generate modeling artifacts needed for app

In [ ]:
if SKIP_TRAINING:
    # Extract configuration from existing model for artifacts generation
    print("Extracting configuration from existing model...")

    # Get project details
    project = dr.Project.get(project_id)
    model_obj = dr.Model.get(project=project_id, model_id=model_id)

    # project.target can return "X (actual)" for time series projects; strip the suffix
    # so downstream code that builds column names like f"{target}_PREDICTION" works correctly
    target = project.target.removesuffix(" (actual)")
    datetime_partition_column = next(
        (
            feature.name
            for feature in project.get_features()
            if feature.feature_type == "Date"
        ),
        "Date",  # fallback
    )

    # Create a simplified autopilotrun_args for compatibility
    from datarobot_pulumi_utils.schema.training import AnalyzeAndModelArgs

    from forecastic.schema import FeatureSettingConfig

    # Update feature_settings_config to match your model's known-in-advance features
    autopilotrun_args = type(
        "AutopilotRunArgs",
        (),
        {
            "analyze_and_model_config": AnalyzeAndModelArgs(
                target=target,
                metric="RMSE",
                mode=dr.enums.AUTOPILOT_MODE.QUICK,
                worker_count=-1,
            ),
            "feature_settings_config": [
                FeatureSettingConfig(feature_name="headcount", known_in_advance=True),
                FeatureSettingConfig(
                    feature_name="renewal_rate_pct", known_in_advance=True
                ),
                FeatureSettingConfig(feature_name="industry", known_in_advance=True),
                FeatureSettingConfig(feature_name="sector", known_in_advance=True),
                FeatureSettingConfig(feature_name="area", known_in_advance=True),
                FeatureSettingConfig(
                    feature_name="business_unit", known_in_advance=True
                ),
                FeatureSettingConfig(feature_name="region", known_in_advance=True),
            ],
        },
    )()

    print(f"Using target: {target}")
    print(f"Using datetime partition column: {datetime_partition_column}")

In [ ]:
from forecastic.schema import WhatIfFeature


def get_what_if_features(
    project_id: str,
    model_id: str,
    feature_settings_config: Optional[List[FeatureSettingConfig]] = None,
) -> List[WhatIfFeature]:
    """Returns features to be exposed in app for what if analysis

    Only returns categorical and numeric known in advance features.
    Categories are returned with selectable options.

    Parameters
    ----------
    feature_settings_config : Optional[List[Dict[str, Any]]]
        Known in advance features
    """

    if not feature_settings_config:
        return []

    project = dr.Project.get(project_id)  # type: ignore[attr-defined]
    model = dr.Model.get(project=project_id, model_id=model_id)  # type: ignore[attr-defined]
    dataset = project.get_dataset()
    if dataset is None:
        raise ValueError("Dataset not found")
    model_features = set(model.get_features_used())
    feature_types = dataset.get_all_features()
    dataframe = dataset.get_as_dataframe()

    numerics = set([i.name for i in feature_types if i.feature_type == "Numeric"])
    categoricals = set(
        [i.name for i in feature_types if i.feature_type == "Categorical"]
    )
    allowed_features = numerics.union(categoricals)

    whatif_features = []
    for feature in feature_settings_config:
        if (
            feature.known_in_advance
            and feature.feature_name in model_features
            and feature.feature_name in allowed_features
        ):
            append_feature = feature.model_dump(mode="json")
            if feature.feature_name in categoricals:
                append_feature["values"] = list(
                    dataframe[feature.feature_name].unique()
                )

            whatif_features.append(WhatIfFeature(**append_feature))
    return whatif_features

In [ ]:
def get_most_important_features(
    project_id: str,
    model_id: str,
    minimum_importance: float = 0.03,
    max_wait: int = 600,
) -> List[Dict[str, Any]]:
    """Get the most important features for the model.

    Parameters
    ----------
    max_features : int
        The maximum number of features to return
    max_wait : int
        The maximum time to wait for the feature impact to be calculated
    """

    model = dr.Model.get(model_id=model_id, project=project_id)  # type: ignore[attr-defined]
    feature_impact = model.get_or_request_feature_impact(max_wait=max_wait)

    return [
        {
            "featureName": feature["featureName"],
            "impactNormalized": feature["impactNormalized"],
        }
        for feature in feature_impact
        if feature["impactNormalized"] > minimum_importance
    ]

In [ ]:
def get_timestep_settings(
    project_id: str,
    datetime_partition_column: str,
) -> Dict[str, Any]:
    """Get window basis unit and interval from timeseries project

    Returns
    -------
    Dict[str, Any]
        Time unit and step
    """
    url = f"projects/{project_id}/features/{datetime_partition_column}/multiseriesProperties"
    response = client.get(url).json()
    timestep_settings: dict[str, Any] = response["detectedMultiseriesIdColumns"][0]
    del timestep_settings["multiseriesIdColumns"]
    return timestep_settings

In [ ]:
print("Running feature impact...")
important_features = get_most_important_features(
    project_id=project_id,
    model_id=model_id,
    minimum_importance=0.05,  # cleanup
)

# Export settings for provisioning app, other dependent resources

In [ ]:
import textwrap

from forecastic.i18n import gettext
from forecastic.schema import CategoryFilter, StaticAppSettings

industry_display_name = gettext("Industry")
sector_display_name = gettext("Sector")
area_display_name = gettext("Area")
business_unit_display_name = gettext("Business Unit")
region_display_name = gettext("Region")
page_description = gettext(
    "This application forecasts services revenue. The forecast can be focused by industry, sector, area, business unit, or region."
)
graph_y_axis = gettext("Revenue ($)")
page_title = (
    f"Forecast Analyst: {deployment.label}"
    if SKIP_TRAINING
    else gettext("Forecast Assistant - Services Revenue")
)
headline_prompt = textwrap.dedent(
    gettext("""\
        You are a data analyst and your job is to explain to non-technical executive business leaders what the data suggests
        Executive leadership will provide a revenue forecast and you will interpret it and summarize the outlook, highlighting key insights.
        Your response should be only 1 sentence long, not very wordy. It should be like a news headline. Do not put quotation marks around it.
        Your response, while insightful, should speak to the general direction of the forecast."""),
)


static_app_settings = StaticAppSettings(
    filterable_categories=[
        CategoryFilter(column_name="industry", display_name=industry_display_name),
        CategoryFilter(column_name="sector", display_name=sector_display_name),
        CategoryFilter(column_name="area", display_name=area_display_name),
        CategoryFilter(
            column_name="business_unit", display_name=business_unit_display_name
        ),
        CategoryFilter(column_name="region", display_name=region_display_name),
    ],
    page_description=page_description,
    lower_bound_forecast_at_0=True,
    graph_y_axis=graph_y_axis,
    page_title=page_title,
    headline_prompt=headline_prompt,
    llm_commentary_enabled=True,
)

In [ ]:
import yaml

from forecastic.schema import AppSettings
from infra.settings_main import model_training_output_file

print("Capturing settings required to deploy the frontend...")
print(f"Using registered model: {registered_model_name} ({registered_model_id})")

# Real-time prediction explanations (used for the AI commentary's feature-driver summary)
# require a one-time initialization per model - unlike batch prediction jobs, which can
# compute explanations on the fly. Without this, real-time scoring calls silently omit
# the EXPLANATION_* columns instead of erroring.
print("Checking prediction explanations initialization...")
try:
    dr.PredictionExplanationsInitialization.get(project_id, model_id)
    print(f"Prediction explanations already initialized for model {model_id}")
except dr.errors.ClientError:
    try:
        dr.PredictionExplanationsInitialization.create(
            project_id, model_id
        ).wait_for_completion()
        print(f"Initialized prediction explanations for model {model_id}")
    except dr.errors.ClientError as e:
        print(
            f"Could not initialize prediction explanations for model {model_id}: {e}. "
            "The app will still work, but the AI commentary won't include feature-driver "
            "explanations until this is resolved."
        )

# Only set a prediction interval if one has actually been pre-computed for this model —
# the deployment API rejects `prediction_intervals_settings` otherwise. The registration
# step above waits for the model package build to complete (including
# compute_all_ts_intervals), so this reflects the real, current state either way.
calculated_intervals = dr.Model.get(
    project=project_id, model_id=model_id
).get_calculated_prediction_intervals()
prediction_interval = (
    80 if 80 in calculated_intervals else next(iter(calculated_intervals), None)
)
if prediction_interval is None:
    print(
        f"No pre-computed prediction intervals found for model {model_id}; "
        "deploying without prediction intervals enabled."
    )

if SKIP_TRAINING:
    if not registered_model_id:
        raise ValueError(
            f"Could not determine registered model ID from deployment model_package. "
            f"Keys available: {list(deployment.model_package.keys())}"
        )

    # Build AppSettings directly from project + deployment metadata.
    # This avoids dr.RegisteredModel.get() which requires registered-model read access
    # and may be restricted in customer environments.
    datetime_partitioning = dr.DatetimePartitioning.get(project_id)
    dt_spec = datetime_partitioning.get_input_data(
        project_id, datetime_partitioning.datetime_partitioning_id
    )

    multiseries_id_columns = dt_spec.multiseries_id_columns or []
    multiseries_id_column = multiseries_id_columns[0] if multiseries_id_columns else None
    filterable_categories = static_app_settings.filterable_categories if multiseries_id_column else []

    training_dataset = project.get_dataset()
    training_dataset_id = training_dataset.id if training_dataset else ""

    app_settings = AppSettings(
        registered_model_id=registered_model_id,
        registered_model_version_id=registered_model_version_id,
        what_if_features=get_what_if_features(
            project_id=project_id,
            model_id=model_id,
            feature_settings_config=autopilotrun_args.feature_settings_config,
        ),
        important_features=important_features,
        prediction_interval=prediction_interval,
        use_case_id=use_case_id,
        project_id=project_id,
        model_id=model_id,
        model_name=dr.Model.get(project_id, model_id).model_type,
        date_format=datetime_partitioning.date_format,
        target=target,
        multiseries_id_column=multiseries_id_column,
        feature_derivation_window_start=datetime_partitioning.feature_derivation_window_start,
        feature_derivation_window_end=datetime_partitioning.feature_derivation_window_end,
        forecast_window_start=datetime_partitioning.forecast_window_start,
        forecast_window_end=datetime_partitioning.forecast_window_end,
        maximum_default_display_length=datetime_partitioning.forecast_window_end * 10,
        timestep_settings=AppSettings.get_timestamp_settings(
            project_id, dt_spec.datetime_partition_column
        ),
        datetime_partition_column=dt_spec.datetime_partition_column,
        datetime_partition_column_transformed=datetime_partitioning.datetime_partition_column,
        training_dataset_id=training_dataset_id,
        calendar_id=datetime_partitioning.calendar_id,
        filterable_categories=filterable_categories,
        page_description=static_app_settings.page_description,
        lower_bound_forecast_at_0=static_app_settings.lower_bound_forecast_at_0,
        graph_y_axis=static_app_settings.graph_y_axis,
        page_title=static_app_settings.page_title,
        headline_prompt=static_app_settings.headline_prompt,
        llm_commentary_enabled=static_app_settings.llm_commentary_enabled,
    )
    print(f"Built app settings from deployment {FORECAST_DEPLOYMENT_ID}")

else:
    # Training path — use the factory method as before. registered_model_id was already
    # resolved above (from the freshly registered model package), so no separate
    # dr.RegisteredModel.list() lookup is needed here.
    app_settings = AppSettings.from_registered_model_version(
        target=autopilotrun_args.analyze_and_model_config.target,
        registered_model_id=registered_model_id,
        registered_model_version_id=registered_model_version_id,
        what_if_features=get_what_if_features(
            project_id=project_id,
            model_id=model_id,
            feature_settings_config=autopilotrun_args.feature_settings_config,
        ),
        important_features=important_features,
        prediction_interval=prediction_interval,
        static_app_settings=static_app_settings,
    )
    print("Built app settings from newly trained model")

with open(model_training_output_file, "w") as f:
    yaml.dump(app_settings.model_dump(), f, allow_unicode=True)

print(f"Wrote {model_training_output_file}")